<a href="https://colab.research.google.com/github/sDeNySoKs/RecomendationSystem/blob/main/RecomendationSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "numpy<2" scikit-surprise

In [2]:
from surprise import Dataset, SVD, SVDpp, NMF
from surprise.model_selection import cross_validate, GridSearchCV

data = Dataset.load_builtin('ml-100k')

Dataset ml-100k could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


In [3]:
print("=== Порівняння Алгоритмів ===")

print("\n1. Крос-валідація SVD:")
algo_svd = SVD(random_state=42)
cross_validate(algo_svd, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

print("\n2. Крос-валідація SVD++:")
algo_svdpp = SVDpp(random_state=42)
cross_validate(algo_svdpp, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

print("\n3. Крос-валідація NMF:")
algo_nmf = NMF(random_state=42)
cross_validate(algo_nmf, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

=== Порівняння Алгоритмів ===

1. Крос-валідація SVD:
Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9450  0.9434  0.9441  0.9441  0.0007  
MAE (testset)     0.7453  0.7452  0.7434  0.7446  0.0009  
Fit time          3.09    2.23    1.03    2.12    0.84    
Test time         1.22    0.39    0.18    0.59    0.45    

2. Крос-валідація SVD++:
Evaluating RMSE, MAE of algorithm SVDpp on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9267  0.9272  0.9289  0.9276  0.0009  
MAE (testset)     0.7270  0.7273  0.7303  0.7282  0.0015  
Fit time          16.95   16.21   16.48   16.54   0.31    
Test time         7.24    7.18    6.21    6.88    0.47    

3. Крос-валідація NMF:
Evaluating RMSE, MAE of algorithm NMF on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9711  0.9760  0.9709  0.9727  0.0024  
MAE (testset)     0.7623

{'test_rmse': array([0.97106553, 0.97602906, 0.97092454]),
 'test_mae': array([0.76226202, 0.76731118, 0.76202757]),
 'fit_time': (2.1459124088287354, 1.5020923614501953, 1.7072761058807373),
 'test_time': (0.22030234336853027, 0.23344993591308594, 0.284099817276001)}

In [4]:
print("=== Підбір гіпермараметрів для SVD ===")

param_grid = {
    'n_epochs': [10, 20], # кількість ітерацій
    'lr_all': [0.002, 0.005], # швидість навчання
    'reg_all': [0.02, 0.1] # Регуляризація
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

print(f"Найкраще значення RMSE: {gs.best_score['rmse']:.4f}")
print(f"Найкращі параметри: {gs.best_params['rmse']}")


=== Підбір гіпермараметрів для SVD ===
Найкраще значення RMSE: 0.9421
Найкращі параметри: {'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.1}


In [5]:
best_algo = gs.best_estimator['rmse']

trainset = data.build_full_trainset()
best_algo.fit(trainset)

user_id = str(196)
item_id = str(302)

prediction = best_algo.predict(user_id, item_id)
print(f"\nПрогноз для користувача {user_id} на фільм {item_id}:")
print(f"Очікувана оцінка: {prediction.est:.2f} (з 5.00)")


Прогноз для користувача 196 на фільм 302:
Очікувана оцінка: 4.10 (з 5.00)


Висновок:
У результаті порівняння трьох алгоритмів матричної факторизації найнижчу похибку (RMSE) продемонстрував алгоритм SVD++, оскільки він додатково враховує неявні взаємодії користувачів (факт оцінювання інших фільмів). Однак алгоритм SVD показав дуже близькі результати точності, працюючи при цьому в рази швидше. За допомогою GridSearchCV для SVD було знайдено оптимальні гіперпараметри, що дозволило побудувати фінальну модель та успішно згенерувати прогноз оцінки для конкретного користувача та фільму. Алгоритм NMF показав найгіршу точність серед протестованих на цьому датасеті.